# Layer Normalization (LayerNorm) — Complete Beginner Masterclass

**Based on:** [What Is Layer Normalization? — GeeksforGeeks](https://www.geeksforgeeks.org/deep-learning/what-is-layer-normalization/)

This Colab notebook goes through the article's topics, works through its three example vectors, shows every calculation, and then compares **no normalization vs Batch Normalization vs Layer Normalization** while training the same type of neural network on MNIST.

## Learning goals

By the end, you should be able to explain:

1. What Layer Normalization does and why it can help training.
2. How to calculate mean, variance, normalized values, and the final output by hand.
3. Why LayerNorm uses **features within one sample**, not other samples in the batch.
4. What the learnable parameters gamma (\(\gamma\)) and beta (\(\beta\)) do.
5. How LayerNorm differs from BatchNorm.
6. How to add `nn.LayerNorm(...)` to a PyTorch model.
7. How to compare actual training loss, accuracy, activation statistics, and timing.
8. Why smoother training curves are possible but **not guaranteed** by normalization.

**Source note:** This notebook summarizes the article in original wording, includes its example vectors and PyTorch structure, and adds an executable MNIST experiment. It does not reproduce the article verbatim.

## 1. What is Layer Normalization?

Layer Normalization (LayerNorm) is a neural-network layer that normalizes the feature values **within each individual sample**.

Suppose one hidden layer produces this activation vector for one sample:

\[
x=[3,\;5,\;2,\;8].
\]

LayerNorm calculates the mean and variance of these four feature values, normalizes each value, and then applies learnable scale and shift parameters.

The usual formula is:

\[
\mu=\frac{1}{H}\sum_{i=1}^{H}x_i
\]

\[
\sigma^2=\frac{1}{H}\sum_{i=1}^{H}(x_i-\mu)^2
\]

\[
\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}
\]

\[
\boxed{y_i=\gamma_i\hat{x}_i+\beta_i}
\]

Here:

- \(H\): number of features in the normalized layer.
- \(\mu\): mean across those features for this sample.
- \(\sigma^2\): variance across those features for this sample.
- \(\epsilon\): small positive constant for numerical stability.
- \(\gamma_i\): learned scale for feature \(i\).
- \(\beta_i\): learned shift for feature \(i\).

In a real LayerNorm layer, gamma and beta are **vectors**: each normalized feature has its own scale and shift.

## 2. LayerNorm vs BatchNorm — the key difference

Consider a batch matrix whose rows are samples and whose columns are features:

\[
X=
\begin{bmatrix}
x_{11}&x_{12}&x_{13}&x_{14}\\
x_{21}&x_{22}&x_{23}&x_{24}\\
x_{31}&x_{32}&x_{33}&x_{34}
\end{bmatrix}.
\]

- **LayerNorm:** for each row, calculate statistics across its feature columns. Each sample is normalized independently.
- **BatchNorm:** for each feature column, calculate statistics across the sample rows in the mini-batch.

The visual in this notebook will highlight different axes:

- LayerNorm: normalize across columns **within a row**.
- BatchNorm: normalize down rows **within a column**.

LayerNorm therefore does not rely on the mini-batch size. BatchNorm's training statistics do depend on the current batch.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# Small illustrative batch: rows are samples; columns are features.
toy_batch = np.array([
    [3.0, 5.0, 2.0, 8.0],
    [1.0, 3.0, 5.0, 8.0],
    [3.0, 2.0, 7.0, 9.0]
])

toy_df = pd.DataFrame(
    toy_batch,
    index=["sample 1", "sample 2", "sample 3"],
    columns=["feature 1", "feature 2", "feature 3", "feature 4"]
)
display(toy_df)

ln_mean = toy_batch.mean(axis=1, keepdims=True)
ln_var = toy_batch.var(axis=1, keepdims=True)  # population variance, ddof=0
bn_mean = toy_batch.mean(axis=0, keepdims=True)
bn_var = toy_batch.var(axis=0, keepdims=True)

print("LayerNorm mean per sample (one per row):")
print(ln_mean.ravel())
print("LayerNorm variance per sample:")
print(ln_var.ravel())
print("\\nBatchNorm mean per feature (one per column):")
print(bn_mean.ravel())
print("BatchNorm variance per feature:")
print(bn_var.ravel())

In [ ]:
# Side-by-side mathematical normalization on the same toy batch
epsilon = 1e-5
toy_ln = (toy_batch - ln_mean) / np.sqrt(ln_var + epsilon)
toy_bn = (toy_batch - bn_mean) / np.sqrt(bn_var + epsilon)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(
    axes,
    [toy_batch, toy_ln, toy_bn],
    ["Original data", "LayerNorm: across each row", "BatchNorm: down each column"]
):
    im = ax.imshow(data, aspect="auto", cmap="coolwarm")
    ax.set_title(title)
    ax.set_xlabel("Features")
    ax.set_ylabel("Samples")
    ax.set_xticks(range(4), ["F1", "F2", "F3", "F4"])
    ax.set_yticks(range(3), ["S1", "S2", "S3"])
    for r in range(data.shape[0]):
        for c in range(data.shape[1]):
            ax.text(c, r, f"{data[r,c]:.2f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("LayerNorm per-row means:", toy_ln.mean(axis=1))
print("LayerNorm per-row variances:", toy_ln.var(axis=1))
print("BatchNorm per-column means:", toy_bn.mean(axis=0))
print("BatchNorm per-column variances:", toy_bn.var(axis=0))

## 3. Reproduce the article's first example, step by step

The article uses these three vectors:

\[
x_1=[3.0,5.0,2.0,8.0]
\]

\[
x_2=[1.0,3.0,5.0,8.0]
\]

\[
x_3=[3.0,2.0,7.0,9.0].
\]

LayerNorm calculates one mean and one variance **for each vector independently**. The number of features is \(H=4\).

For \(x_1\):

\[
\mu_1=\frac{3+5+2+8}{4}=\frac{18}{4}=4.5
\]

\[
\begin{aligned}
\sigma_1^2
&=\frac{(3-4.5)^2+(5-4.5)^2+(2-4.5)^2+(8-4.5)^2}{4}\\
&=\frac{2.25+0.25+6.25+12.25}{4}\\
&=\frac{21}{4}=5.25.
\end{aligned}
\]

Then:

\[
\hat{x}_i=\frac{x_i-\mu_1}{\sqrt{\sigma_1^2+\epsilon}},
\qquad \epsilon=10^{-5}.
\]

We will calculate the individual subtraction, squared deviation, denominator, and normalized feature with Python.

In [ ]:
vectors = {
    "x1": np.array([3.0, 5.0, 2.0, 8.0]),
    "x2": np.array([1.0, 3.0, 5.0, 8.0]),
    "x3": np.array([3.0, 2.0, 7.0, 9.0]),
}
EPS = 1e-5

def layer_norm_calculation(vector, epsilon=EPS):
    mean = vector.mean()
    squared_deviations = (vector - mean) ** 2
    variance = squared_deviations.mean()
    denominator = np.sqrt(variance + epsilon)
    normalized = (vector - mean) / denominator
    return {
        "mean": mean,
        "deviations": vector - mean,
        "squared_deviations": squared_deviations,
        "variance": variance,
        "denominator": denominator,
        "normalized": normalized
    }

x1_details = layer_norm_calculation(vectors["x1"])

print("x1 =", vectors["x1"])
print("Sum =", vectors["x1"].sum())
print("Mean = sum / H =", vectors["x1"].sum(), "/", len(vectors["x1"]), "=", x1_details["mean"])
print("Deviations:", x1_details["deviations"])
print("Squared deviations:", x1_details["squared_deviations"])
print("Sum of squared deviations:", x1_details["squared_deviations"].sum())
print("Variance = sum squared deviations / H =", x1_details["variance"])
print("Denominator sqrt(variance + epsilon) =", x1_details["denominator"])
print("Normalized values =", x1_details["normalized"])

In [ ]:
# Detailed table for all three source vectors
detail_rows = []
for name, vec in vectors.items():
    d = layer_norm_calculation(vec)
    for i, value in enumerate(vec):
        detail_rows.append({
            "Vector": name,
            "Feature index": i + 1,
            "Original x_i": value,
            "Mean for this vector": d["mean"],
            "x_i - mean": d["deviations"][i],
            "(x_i - mean)^2": d["squared_deviations"][i],
            "Variance for this vector": d["variance"],
            "sqrt(variance + eps)": d["denominator"],
            "Normalized x_hat_i": d["normalized"][i]
        })

detail_df = pd.DataFrame(detail_rows)
display(detail_df.round(6))

## 4. Important source check: arithmetic in the article

The article's formulas and example vectors are useful, but the text appears to contain arithmetic errors in the printed variance values for \(x_2\) and \(x_3\).

For the given vectors, the correctly computed values are:

- \(x_1=[3,5,2,8]\): mean \(4.5\), variance \(5.25\).
- \(x_2=[1,3,5,8]\): mean \(4.25\), variance **\(6.6875\)**.
- \(x_3=[3,2,7,9]\): mean \(5.25\), variance **\(8.1875\)**.

The article's written variance values for \(x_2\) and \(x_3\) do not match the sums of squared deviations. Its stated normalized vectors for those two examples are consistent with the corrected variances above. This notebook uses the arithmetic that follows directly from the formulas and data.

In [ ]:
variance_check_rows = []
for name, vec in vectors.items():
    mean = np.mean(vec)
    deviations = vec - mean
    squares = deviations ** 2
    variance = np.mean(squares)
    variance_check_rows.append({
        "Vector": name,
        "Vector": name,
        "Mean": mean,
        "Deviations": np.array2string(deviations, precision=4),
        "Squared deviations": np.array2string(squares, precision=4),
        "Sum of squared deviations": squares.sum(),
        "Correct variance": variance
    })

display(pd.DataFrame(variance_check_rows).set_index("Vector").round(6))

## 5. Apply gamma and beta: scale and shift

After normalization, LayerNorm applies:

\[
y_i=\gamma_i\hat{x}_i+\beta_i.
\]

The article illustrates this with scalar values \(\gamma=1.5\) and \(\beta=0.5\). We'll use those same scalar values for the worked calculation to make the arithmetic easy to follow.

For example, for the first normalized value of \(x_1\):

\[
y_1=1.5(-0.6547)+0.5\approx-0.4821.
\]

In a real PyTorch layer with `nn.LayerNorm(4)`, \(\gamma\) and \(\beta\) are vectors of length four, so each feature can learn a distinct scale and shift.

In [ ]:
gamma_scalar = 1.5
beta_scalar = 0.5
scaled_rows = []

for name, vec in vectors.items():
    d = layer_norm_calculation(vec)
    output = gamma_scalar * d["normalized"] + beta_scalar
    for i in range(len(vec)):
        scaled_rows.append({
            "Vector": name,
            "Feature": i + 1,
            "Normalized x_hat": d["normalized"][i],
            "gamma * x_hat": gamma_scalar * d["normalized"][i],
            "beta": beta_scalar,
            "Final y": output[i]
        })

display(pd.DataFrame(scaled_rows).round(6))

In [ ]:
# Compare normalized activations with several gamma/beta settings
x1_norm = x1_details["normalized"]
settings = [
    (1.0, 0.0, "gamma=1, beta=0"),
    (1.5, 0.5, "gamma=1.5, beta=0.5"),
    (0.5, -1.0, "gamma=0.5, beta=-1")
]

plt.figure(figsize=(9, 4))
features = np.arange(1, 5)
for gamma, beta, label in settings:
    plt.plot(features, gamma * x1_norm + beta, marker="o", label=label)
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("Feature index")
plt.ylabel("Output")
plt.title("How gamma and beta change normalized values")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. How LayerNorm works inside a neural network

A typical feed-forward block in the article is:

\[
\text{Linear} \rightarrow \text{LayerNorm} \rightarrow \text{ReLU}
\]

- `nn.Linear(input_size, 128)` creates a fully connected transformation.
- `nn.LayerNorm(128)` normalizes the 128 features independently for each sample.
- `torch.relu(x)` applies ReLU element-wise.
- A final linear layer produces class logits.

Unlike BatchNorm, LayerNorm does not need to store running means and variances for inference: the normalization statistics are calculated from each input sample in both training and evaluation modes.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class SimpleLayerNormNN(nn.Module):
    def __init__(self, input_size=64, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.layer_norm = nn.LayerNorm(128)
        self.fc2 = nn.Linear(128, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.layer_norm(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

input_data = torch.randn(10, 64)
model_example = SimpleLayerNormNN(input_size=64, output_size=10)
output = model_example(input_data)

print("Input shape:", input_data.shape)
print("Output shape:", output.shape)
print("First sample logits:", output[0].detach().numpy().round(4))
print("Learnable LayerNorm gamma shape:", tuple(model_example.layer_norm.weight.shape))
print("Learnable LayerNorm beta shape:", tuple(model_example.layer_norm.bias.shape))

## 7. Auto-download MNIST for the training comparison

Now we'll train three comparable models on the same downloaded dataset:

1. **No normalization**
2. **Batch Normalization**
3. **Layer Normalization**

MNIST contains 28×28 grayscale digit images from 10 classes. PyTorch's `torchvision.datasets.MNIST(..., download=True)` downloads the dataset automatically in Colab.

We will convert the images to float tensors in \([0,1]\), flatten each image into 784 inputs, and use the same train/test subsets for every model.

This gives an educational comparison of real training curves. A single run is not a universal scientific verdict: results depend on model, optimizer, learning rate, batch size, random seed, and training duration.

In [ ]:
import time
import random
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Torchvision dataset will download automatically if needed.")
print("Device:", device)

transform = transforms.Compose([
    transforms.ToTensor(),  # Converts uint8 pixels [0,255] to float [0,1]
])

train_dataset_full = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset_full = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# Use a reproducible subset to make the notebook practical for Colab.
# Increase these limits if you want a longer experiment.
NUM_TRAIN = 12000
NUM_TEST = 3000

rng = np.random.default_rng(SEED)
train_indices = rng.permutation(len(train_dataset_full))[:NUM_TRAIN].tolist()
test_indices = rng.permutation(len(test_dataset_full))[:NUM_TEST].tolist()

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

BATCH_SIZE = 128
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i, 0].numpy(), cmap="gray")
    plt.title(f"Label: {labels[i].item()}")
    plt.axis("off")
plt.suptitle("Examples from auto-downloaded MNIST")
plt.tight_layout()
plt.show()

## 8. Build equivalent models: no normalization, BatchNorm, and LayerNorm

The same multilayer perceptron structure is used for every experiment:

\[
784 \rightarrow 256 \rightarrow 128 \rightarrow 64 \rightarrow 10.
\]

For the normalized variants, each hidden block is:

\[
\text{Linear} \rightarrow \text{Normalization} \rightarrow \text{ReLU}.
\]

- `BatchNorm1d(width)` uses mini-batch statistics across samples for each hidden feature during training.
- `LayerNorm(width)` uses feature statistics independently for each sample.
- The baseline omits the normalization layer.

All models use the same optimizer, learning rate, data split, batch size, epoch count, and loss function. Model initialization is seeded for reproducibility, although the models are not guaranteed to start from exactly identical weights because normalization changes parameter shapes and initialization calls.

In [ ]:
class MLP(nn.Module):
    def __init__(self, norm_type="none"):
        super().__init__()
        if norm_type not in {"none", "batch", "layer"}:
            raise ValueError("norm_type must be 'none', 'batch', or 'layer'")
        self.norm_type = norm_type

        modules = []
        widths = [784, 256, 128, 64]

        for in_features, out_features in zip(widths[:-1], widths[1:]):
            modules.append(nn.Linear(in_features, out_features))
            if norm_type == "batch":
                modules.append(nn.BatchNorm1d(out_features))
            elif norm_type == "layer":
                modules.append(nn.LayerNorm(out_features))
            modules.append(nn.ReLU())

        modules.append(nn.Linear(widths[-1], 10))
        self.network = nn.Sequential(*modules)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)  # logits; CrossEntropyLoss applies log-softmax internally

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for norm_type in ["none", "batch", "layer"]:
    torch.manual_seed(SEED)
    model = MLP(norm_type)
    print(f"{norm_type:>5} normalization: {count_parameters(model):,} trainable parameters")

## 9. Training and evaluation functions

We'll use cross-entropy loss with Adam.

For each training batch:

1. Run the forward pass.
2. Calculate the loss.
3. Clear old gradients.
4. Backpropagate the loss.
5. Update model parameters.

For evaluation, `model.eval()` activates the appropriate inference behaviour. BatchNorm uses its running statistics, while LayerNorm continues to normalize each sample across its features.

We will record training loss, test loss, training accuracy, test accuracy, and epoch duration.

In [ ]:
import torch.nn.functional as F

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

def train_model(norm_type, epochs=5, learning_rate=1e-3):
    torch.manual_seed(SEED)
    model = MLP(norm_type).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "test_loss": [],
        "test_accuracy": [],
        "epoch_seconds": []
    }

    for epoch in range(epochs):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_accuracy = correct / total
        test_loss, test_accuracy = evaluate_model(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["test_loss"].append(test_loss)
        history["test_accuracy"].append(test_accuracy)
        history["epoch_seconds"].append(time.time() - epoch_start)

        print(
            f"{norm_type:>5} | Epoch {epoch+1:02d}/{epochs} | "
            f"train loss {train_loss:.4f} | train acc {train_accuracy*100:.2f}% | "
            f"test loss {test_loss:.4f} | test acc {test_accuracy*100:.2f}% | "
            f"{history['epoch_seconds'][-1]:.1f}s"
        )

    return model, history

## 10. Run the comparison

Start with five epochs. The subset sizes make this practical in Colab while keeping the comparison meaningful.

If a GPU is available, training will generally be faster. On CPU it may take longer. You can lower `NUM_TRAIN`, `NUM_TEST`, or `EPOCHS` to test the notebook quickly, then increase them for a more stable comparison.

**Do not assume LayerNorm must win.** The experiment records the real outcomes from your runtime.

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3

models = {}
histories = {}
overall_start = time.time()

for norm_type in ["none", "batch", "layer"]:
    print("\\n" + "=" * 88)
    print("Training model with:", norm_type, "normalization")
    model, history = train_model(
        norm_type=norm_type,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE
    )
    models[norm_type] = model
    histories[norm_type] = history

print(f"\\nTotal comparison time: {(time.time() - overall_start)/60:.2f} minutes")

## 11. Compare the real training curves

Look at both training loss and test accuracy. A curve that decreases smoothly can be easier to optimize, but smoothness alone is not the same as better generalization. Also compare final test loss and accuracy.

The plots below are generated directly from the measurements recorded during training.

In [ ]:
epochs_axis = np.arange(1, EPOCHS + 1)
labels = {
    "none": "No normalization",
    "batch": "Batch Normalization",
    "layer": "Layer Normalization"
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for key in ["none", "batch", "layer"]:
    axes[0].plot(epochs_axis, histories[key]["train_loss"], marker="o", label=labels[key])
    axes[0].plot(
        epochs_axis, histories[key]["test_loss"],
        marker=".", linestyle="--", alpha=0.75, label=labels[key] + " (test)"
    )
axes[0].set_title("Loss across epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

for key in ["none", "batch", "layer"]:
    axes[1].plot(
        epochs_axis,
        np.array(histories[key]["test_accuracy"]) * 100,
        marker="o",
        label=labels[key]
    )
axes[1].set_title("Test accuracy across epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
summary_rows = []
for key in ["none", "batch", "layer"]:
    h = histories[key]
    summary_rows.append({
        "Normalization": labels[key],
        "Final train loss": h["train_loss"][-1],
        "Final train accuracy (%)": 100 * h["train_accuracy"][-1],
        "Final test loss": h["test_loss"][-1],
        "Final test accuracy (%)": 100 * h["test_accuracy"][-1],
        "Total epoch time (s)": sum(h["epoch_seconds"]),
        "Average epoch time (s)": np.mean(h["epoch_seconds"])
    })

comparison_df = pd.DataFrame(summary_rows).sort_values(
    "Final test accuracy (%)", ascending=False
)
display(comparison_df.round(4))

## 12. What does “smoother training” mean?

A normalization layer may make training more stable, but we should define what we mean rather than merely eyeballing the curves.

For this simple experiment, we'll report:
- Whether training loss decreased at each epoch.
- The variation in epoch-to-epoch loss changes.
- The overall trend and test performance.

Only five epochs is a short demonstration. A model can have a small loss wiggle and still be training well, and a smooth curve can still overfit. More seeds and longer runs would be needed for a rigorous benchmark.

In [ ]:
smoothness_rows = []
for key in ["none", "batch", "layer"]:
    losses = np.array(histories[key]["train_loss"], dtype=float)
    differences = np.diff(losses)
    smoothness_rows.append({
        "Normalization": labels[key],
        "Epochs with lower train loss": int(np.sum(differences < 0)),
        "Epoch-to-epoch loss-change std": float(np.std(differences)),
        "First train loss": float(losses[0]),
        "Last train loss": float(losses[-1]),
        "Total train-loss change": float(losses[-1] - losses[0])
    })

display(pd.DataFrame(smoothness_rows).round(5))

## 13. Compare activation statistics inside the trained networks

The core distinction remains:

- LayerNorm computes mean and variance **within each sample, over features**.
- BatchNorm computes mean and variance **over samples in the mini-batch, for each feature** during training.

To make this measurable, we'll capture the output of the first hidden normalization layer for a fixed batch and examine its statistics.

In [ ]:
fixed_images, fixed_labels = next(iter(test_loader))
fixed_images = fixed_images.to(device)

def get_first_normalization_output(model, norm_type, images):
    model.eval()
    captured = {}

    def hook(module, inputs, output):
        captured["output"] = output.detach()

    target_module = None
    for module in model.modules():
        if norm_type == "batch" and isinstance(module, nn.BatchNorm1d):
            target_module = module
            break
        if norm_type == "layer" and isinstance(module, nn.LayerNorm):
            target_module = module
            break

    if target_module is None:
        return None

    handle = target_module.register_forward_hook(hook)
    with torch.no_grad():
        _ = model(images)
    handle.remove()
    return captured.get("output")

layernorm_out = get_first_normalization_output(models["layer"], "layer", fixed_images)
batchnorm_out = get_first_normalization_output(models["batch"], "batch", fixed_images)

if layernorm_out is not None:
    # For LayerNorm, measure statistics over features for each sample.
    print("LayerNorm: first 3 sample means:", layernorm_out.mean(dim=1)[:3].cpu().numpy())
    print("LayerNorm: first 3 sample variances:", layernorm_out.var(dim=1, unbiased=False)[:3].cpu().numpy())

if batchnorm_out is not None:
    # In eval mode BatchNorm uses running statistics, so output over this test batch
    # does not have to have exact mean 0/variance 1 per feature.
    print("\\nBatchNorm (eval mode): first 8 feature means over this test batch:")
    print(batchnorm_out.mean(dim=0)[:8].cpu().numpy())
    print("BatchNorm (eval mode): first 8 feature variances over this test batch:")
    print(batchnorm_out.var(dim=0, unbiased=False)[:8].cpu().numpy())

## 14. Look at LayerNorm's learned parameters

PyTorch `nn.LayerNorm(width)` normally has two trainable vectors:

- `weight`, corresponding to \(\gamma\), initialized to 1.
- `bias`, corresponding to \(\beta\), initialized to 0.

These values are updated by gradient descent. LayerNorm computes its per-sample statistics during both training and evaluation; it does not maintain BatchNorm-style running mean and variance buffers.

In [ ]:
first_ln = next(module for module in models["layer"].modules() if isinstance(module, nn.LayerNorm))
gamma = first_ln.weight.detach().cpu().numpy()
beta = first_ln.bias.detach().cpu().numpy()

print("Gamma shape:", gamma.shape)
print("Beta shape:", beta.shape)
print("First 12 gamma values:", gamma[:12].round(4))
print("First 12 beta values:", beta[:12].round(4))

plt.figure(figsize=(10, 4))
plt.plot(gamma, label="Learned gamma")
plt.plot(beta, label="Learned beta")
plt.title("Learnable LayerNorm scale and shift after training")
plt.xlabel("Hidden feature index")
plt.ylabel("Parameter value")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 15. Layer Normalization advantages and limitations

### Advantages covered by the article

- **Independent of batch size:** every sample is normalized independently, which is useful when batches are small or variable.
- **Training stability:** can help maintain more predictable activation scales and gradient behaviour.
- **Sequential models:** often useful in recurrent networks and Transformers.
- **Consistent sample-wise behavior:** a sample does not depend on which other samples happen to share its batch.

### Limitations covered by the article

- It does not improve every architecture or task.
- Normalization can alter information carried by the original feature scale.
- It adds computations and trainable parameters.
- It is not a universal replacement for BatchNorm; BatchNorm can work very well in conventional CNNs.

### Typical applications

- RNNs, LSTMs, and GRUs.
- Transformers such as BERT- and GPT-style models.
- Some generative architectures.
- Speech and sequence modeling.

These examples identify common use cases; they do not imply that every model in these families uses LayerNorm in exactly the same configuration.

## 16. LayerNorm vs BatchNorm — final comparison

| Property | LayerNorm | BatchNorm |
|---|---|---|
| Axes for a dense layer | Features within each sample | Samples within each feature |
| Depends on batch size? | No | Training statistics depend on batch |
| Running mean / variance? | No | Yes, commonly used for inference |
| Small/variable batches | Often convenient | Can be less reliable with very small batches |
| Typical use | Transformers, RNNs, sequential models | CNNs, conventional feed-forward networks |
| Learnable scale and shift | Yes, per normalized feature | Yes, per feature/channel |

**Remember:** both normalize activations, but they normalize along different dimensions and have different training/inference behaviour.

## 17. Knowledge check

Try answering these before running the answer cell:

1. For \(x=[3,5,2,8]\), calculate its mean and variance.
2. In a tensor shaped `(batch_size, hidden_features)`, which axis does LayerNorm normalize?
3. Why are \(\gamma\) and \(\beta\) learnable?
4. Does LayerNorm use the other samples in a mini-batch to normalize a given sample?
5. Does LayerNorm maintain running means and variances like BatchNorm?
6. Why can LayerNorm be convenient for small batches?
7. Does LayerNorm guarantee faster training or higher accuracy for every network?

In [ ]:
print("1. Mean = 4.5; population variance = 5.25.")
print("2. LayerNorm normalizes across hidden_features for each sample.")
print("3. Gamma scales normalized features; beta shifts them.")
print("4. No. Its statistics are calculated independently for each sample.")
print("5. No. LayerNorm recomputes statistics from each sample in training and evaluation.")
print("6. Its normalization statistics do not depend on batch composition or batch size.")
print("7. No. Benefits depend on architecture, task, optimization settings, and data.")

## 18. Final recap

For one sample with \(H\) features:

\[
\mu=\frac{1}{H}\sum_{i=1}^{H}x_i
\]

\[
\sigma^2=\frac{1}{H}\sum_{i=1}^{H}(x_i-\mu)^2
\]

\[
\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}
\]

\[
\boxed{y_i=\gamma_i\hat{x}_i+\beta_i}
\]

The most important lesson is **which values contribute to the statistics**:

- LayerNorm: the features of each sample.
- BatchNorm: the samples of each mini-batch, separately for each feature during training.

This notebook follows the GeeksforGeeks article's vectors, computation steps, gamma/beta explanation, PyTorch structure, advantages, applications, and limitations. It adds a transparent arithmetic audit, visual examples, and an automatically downloaded MNIST experiment comparing models with no normalization, BatchNorm, and LayerNorm.

### Reference

[GeeksforGeeks — What Is Layer Normalization?](https://www.geeksforgeeks.org/deep-learning/what-is-layer-normalization/)